# Day 1 - Module 1: LLM Foundations & Structured Outputs
### Boeing BCAI Edition - using `BoeingChatModel` (no direct OpenAI calls)

**What you'll build in this notebook:**
1. A working chat call through `BoeingChatModel`
2. Prompt engineering patterns (zero-shot, few-shot, role prompting)
3. Structured output via `with_structured_output()` (schema-enforced JSON)
4. A production-style wrapper with retries, fallback model, and timing

**Estimated time:** 3.5-4.5 hrs (Foundations 2-2.5 hr + Structured Outputs Lab 1.5-2 hr)

> Every LLM call in this notebook goes through `BoeingChatModel`, Boeing's LangChain-compatible
> wrapper around the internal BCAI API. There is no `import openai` anywhere in this notebook.


## 0. Setup

Upload `boeing_chat_model.py` and `boeing_embeddings.py` to this Colab session (left sidebar → Files → upload),
or mount Drive if you keep them there. Then store your BCAI token as a Colab secret named `UDAL_PAT`
(left sidebar → key icon → Secrets).


In [ ]:
# Install dependencies
!pip install -q langchain-core pydantic httpx requests


In [ ]:
# Upload boeing_chat_model.py and boeing_embeddings.py using the Colab file upload widget,
# then confirm both files are present before importing.
import os
from google.colab import files

required = ["boeing_chat_model.py", "boeing_embeddings.py"]
missing = [f for f in required if not os.path.exists(f)]

if missing:
    print(f"Missing: {missing}. Please upload them now.")
    uploaded = files.upload()

for f in required:
    assert os.path.exists(f), f"{f} still missing - upload it before continuing."

print("Wrapper files present:", required)


In [ ]:
# Pull the BCAI token from Colab Secrets (never hardcode tokens in a notebook)
from google.colab import userdata

UDAL_PAT = userdata.get('UDAL_PAT')
assert UDAL_PAT, "Add a Colab secret named UDAL_PAT (key icon in the left sidebar) before continuing."
print("UDAL_PAT loaded:", UDAL_PAT[:4] + "..." + UDAL_PAT[-4:])


In [ ]:
from boeing_chat_model import BoeingChatModel
from boeing_embeddings import BoeingEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Standard chat model - gpt-4.1-mini is the default 'model' field on BoeingChatModel
llm = BoeingChatModel(
    udal_pat=UDAL_PAT,
    model="gpt-4.1-mini",
    temperature=0.2,
    max_tokens=500,
)

print("BoeingChatModel ready:", llm._llm_type, "| model =", llm.model)


## 1. Your first call

Every call below uses LangChain's standard message types (`SystemMessage`, `HumanMessage`, `AIMessage`) -
this is what makes `BoeingChatModel` a drop-in replacement for `ChatOpenAI` in any LangChain pipeline or agent.


In [ ]:
response = llm.invoke([
    SystemMessage(content="You are a concise assistant for enterprise engineers."),
    HumanMessage(content="In two sentences, explain what Retrieval-Augmented Generation (RAG) is."),
])

print(response.content)
print("\nToken usage:", response.response_metadata.get("usage"))


## 2. Prompt engineering patterns

### 2.1 Zero-shot vs. few-shot

Zero-shot: no examples, just an instruction. Few-shot: a small number of input/output examples steer the
model toward a specific output format or style - useful when you need very consistent formatting without
paying for structured-output tool calls.


In [ ]:
zero_shot = llm.invoke([
    HumanMessage(content="Classify the sentiment of this review: 'The onboarding was confusing and slow.'")
])
print("Zero-shot:\n", zero_shot.content)


In [ ]:
few_shot_prompt = '''Classify sentiment as POSITIVE, NEGATIVE, or NEUTRAL. Respond with only the label.

Review: "The install took two minutes and just worked."
Label: POSITIVE

Review: "Documentation was outdated and the API kept timing out."
Label: NEGATIVE

Review: "The onboarding was confusing and slow."
Label:'''

few_shot = llm.invoke([HumanMessage(content=few_shot_prompt)])
print("Few-shot:\n", few_shot.content)


### 2.2 Role / persona prompting

System messages set persistent behavior - tone, constraints, domain framing - separate from the user's
actual question. This is the same `SystemMessage` mechanism used by every LangChain-compatible model.


In [ ]:
role_response = llm.invoke([
    SystemMessage(content=(
        "You are a senior avionics software reviewer. Be precise, flag ambiguity, "
        "and never speculate about certification status."
    )),
    HumanMessage(content="Summarize the tradeoffs between polling and event-driven telemetry ingestion."),
])
print(role_response.content)


### 2.3 Chain-of-thought style reasoning prompts

Asking the model to reason step by step before answering tends to improve accuracy on multi-step problems.
With `BoeingChatModel`, this is just prompt text - no special API parameter needed for standard models.


In [ ]:
cot_response = llm.invoke([
    HumanMessage(content=(
        "A support queue receives 240 tickets/day. 35% are auto-resolved by a bot. "
        "Of the remainder, 20% are escalated to Tier 2. How many tickets/day reach Tier 2? "
        "Think step by step, then give the final number on its own line."
    ))
])
print(cot_response.content)


## 3. Structured outputs - schema-enforced JSON

Free-text responses are hard to parse reliably in production. `with_structured_output()` binds a Pydantic
schema as a *required* tool call - the Boeing API is instructed to always call that tool, and the response is
parsed straight into a validated Python object. Internally this uses `bind_tools(..., tool_choice="required")`.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class TicketTriage(BaseModel):
    """Structured triage result for a single support ticket."""
    category: Literal["billing", "technical", "access", "other"] = Field(
        description="Best-fit category for the ticket."
    )
    priority: Literal["low", "medium", "high", "urgent"] = Field(
        description="Priority based on business impact and urgency language."
    )
    summary: str = Field(description="One-sentence summary of the issue.")
    requires_escalation: bool = Field(description="True if this needs Tier 2 or above.")

structured_llm = llm.with_structured_output(TicketTriage)

result = structured_llm.invoke([
    HumanMessage(content=(
        "Ticket: 'Our production API key stopped authenticating at 2am and our entire "
        "billing sync pipeline has been down since. This is blocking invoicing for 40 customers.'"
    ))
])

print(type(result))
print(result)


In [ ]:
# Inspect it as a validated Pydantic object
print("Category:      ", result.category)
print("Priority:      ", result.priority)
print("Escalate?:     ", result.requires_escalation)
print("Summary:       ", result.summary)


### 3.1 `include_raw=True` - keep the raw model response alongside the parsed object

Useful in production when you want to log token usage or debug a parse failure without losing the raw call.
`with_structured_output(..., include_raw=True)` returns a plain dict of two Runnables
(`{"raw": ..., "parsed": ...}`), not a single invocable object -- wrap it in `RunnableParallel`
so both branches run against the same input in one `.invoke()` call.


In [ ]:
from langchain_core.runnables import RunnableParallel

structured_chain_raw = RunnableParallel(
    **llm.with_structured_output(TicketTriage, include_raw=True)
)

raw_and_parsed = structured_chain_raw.invoke([
    HumanMessage(content="Ticket: 'How do I reset my password? Not urgent.'")
])

print("Keys:", list(raw_and_parsed.keys()))
print("Parsed:", raw_and_parsed["parsed"])


## 4. Production-style wrapper: retries, fallback model, timing

`BoeingChatModel` already retries transport-level failures internally (`max_retries`, exponential backoff -
see `_generate` in `boeing_chat_model.py`). This section adds an *application-level* fallback: if the primary
model instance fails after its own retries are exhausted, fall back to a second, cheaper/more available model.


In [ ]:
import time
from langchain_core.messages import BaseMessage
from typing import List

primary = BoeingChatModel(udal_pat=UDAL_PAT, model="gpt-4.1-mini", temperature=0.2, max_retries=3)
fallback = BoeingChatModel(udal_pat=UDAL_PAT, model="gpt-4.1-mini", temperature=0.0, max_retries=2)

def call_with_fallback(messages: List[BaseMessage], primary_llm=primary, fallback_llm=fallback):
    start = time.time()
    try:
        resp = primary_llm.invoke(messages)
        print(f"[primary succeeded in {time.time() - start:.2f}s]")
        return resp
    except Exception as e:
        print(f"[primary failed: {e} - falling back]")
        start = time.time()
        resp = fallback_llm.invoke(messages)
        print(f"[fallback succeeded in {time.time() - start:.2f}s]")
        return resp

resp = call_with_fallback([HumanMessage(content="Say hello in one short sentence.")])
print(resp.content)


## 5. Wrap-up

You now have a reusable pattern: `BoeingChatModel` for all chat calls, `with_structured_output()` for anything
that needs to be parsed downstream, and an application-level fallback for resilience. The next notebook
(**RAG Fundamentals**) reuses this same `llm` object plus `BoeingEmbeddings` (`text-embedding-3-small`) to build
a retrieval pipeline.

**Checkpoint - you should be able to:**
- Call `BoeingChatModel` with system/human messages
- Explain zero-shot vs. few-shot vs. role prompting
- Bind a Pydantic schema with `with_structured_output()` and get back a validated object
- Explain why application-level fallback differs from the wrapper's built-in retry/backoff
